# TB CXR Diagnostic — Phase 1 (Kaggle)

EfficientNetV2-B0 · domain augmentation · multi-source data

**Before running:** Runtime → GPU T4 x1 (or P100)

## 0. Install

In [ ]:
!pip install -q timm

## 1. Check input paths

Run this first — then paste the correct paths into Config.

In [ ]:
from pathlib import Path

for d in sorted(Path('/kaggle/input').iterdir()):
    children = sorted(d.iterdir())
    print(f'{d}/')
    for c in children[:5]:
        print(f'  {c.name}{"/" if c.is_dir() else ""}')
    if len(children) > 5:
        print(f'  ... ({len(children)} total)')

## 2. Config

In [ ]:
# ── Dataset paths — update after checking cell 1 ────────────────────────────
TBX11K     = '/kaggle/input/datasets/nithish232005/tbx11k/tbx11k'          # set to wherever tbx11k/ folder is
RAHMAN     = '/kaggle/input/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset/TB_Chest_Radiography_Database'
CHEXPERT   = '/kaggle/input/datasets/ashery/chexpert'
# Set to None if not uploaded separately (Rahman already contains both)
SHENZHEN   = None
MONTGOMERY = None
# Set 'montgomery' or 'shenzhen' ONLY when using separate datasets above
HELD_OUT   = 'none'

# ── CheXpert sampling ─────────────────────────────────────────────────────────
CHEXPERT_N_SAMPLES = 10_000   # No-Finding frontal images used as negatives

# ── Model ─────────────────────────────────────────────────────────────────────
# Correct timm name for EfficientNetV2-B0 (tf_ prefix required)
BACKBONE = 'tf_efficientnetv2_b0'

# ── Class imbalance ───────────────────────────────────────────────────────────
# WeightedRandomSampler (always on) already balances batches to ~50/50.
# With balanced batches, pos_weight=1.0 is correct — avoid double-counting.
# Raise to 2-3 only if sensitivity stays too low after a full run.
pos_weight = 1.0

# ── Training ──────────────────────────────────────────────────────────────────
IMAGE_SIZE      = 512
BATCH_SIZE      = 32
EPOCHS          = 50
LR              = 3e-4
WEIGHT_DECAY    = 1e-4
LABEL_SMOOTHING = 0.1
PATIENCE        = 10
SEED            = 42
NUM_WORKERS     = 2

# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUT_DIR          = '/kaggle/working/tb_diagnostic'
SPLITS_CACHE        = '/kaggle/working/tb_diagnostic/splits_cache.pkl'
FORCE_REBUILD_SPLITS = False   # set True to ignore cache and rebuild from scratch

## 3. Imports

In [ ]:
import json, pickle, random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.multiprocessing as _mp
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import functional as TF
from PIL import Image
from sklearn.metrics import roc_auc_score
from tqdm.notebook import tqdm
import timm

_mp.set_start_method('fork', force=True)

print(f'PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  VRAM: {p.total_memory/1e9:.1f} GB')

# Verify backbone name
available_v2 = timm.list_models(f'*{BACKBONE}*')
if available_v2:
    print(f'Backbone ok: {BACKBONE}')
else:
    print(f'WARNING — {BACKBONE!r} not found in timm. '
          f'Available EfficientNetV2 models:')
    print(timm.list_models('*efficientnetv2*')[:10])

## 4. Domain augmentation

Simulates scanner-to-scanner variation in contrast, exposure, and sharpness —
the main driver of cross-source AUC drop.

In [ ]:
class RandomCLAHE:
    """Adaptive histogram equalisation — simulates contrast differences across scanners."""
    def __init__(self, p=0.5, clip_limit=(1.0, 3.0)):
        self.p, self.clip_limit = p, clip_limit

    def __call__(self, img: Image.Image) -> Image.Image:
        if random.random() > self.p:
            return img
        arr   = np.array(img.convert('L'))
        clahe = cv2.createCLAHE(clipLimit=random.uniform(*self.clip_limit),
                                 tileGridSize=(8, 8))
        return Image.fromarray(clahe.apply(arr)).convert('RGB')


class RandomGamma:
    """Gamma shift — simulates different X-ray exposure settings."""
    def __init__(self, p=0.5, gamma_range=(0.7, 1.4)):
        self.p, self.gamma_range = p, gamma_range

    def __call__(self, img: Image.Image) -> Image.Image:
        if random.random() > self.p:
            return img
        return TF.adjust_gamma(img, random.uniform(*self.gamma_range))


def get_train_transforms(size: int):
    return transforms.Compose([
        transforms.Resize((size, size)),
        RandomCLAHE(p=0.5),
        RandomGamma(p=0.5),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.3),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

def get_val_transforms(size: int):
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

## 5. Dataset loaders

In [ ]:
# ── TBX11K — Supervisely per-image JSON tags ───────────────────────────────────
_TBX11K_MAP = {
    'healthy': 0, 'sick_but_non-tb': 0,
    'active_tb': 1, 'active&latent_tb': 1,
    'latent_tb': None, 'no_tag': None,
}

def load_tbx11k(root, split='train', latent_as_positive=False):
    root = Path(root)
    img_dir, ann_dir = root / split / 'img', root / split / 'ann'
    label_map = dict(_TBX11K_MAP)
    if latent_as_positive:
        label_map['latent_tb'] = 1
    samples = []
    for img_path in sorted(img_dir.glob('*.png')):
        ann_path = ann_dir / f'{img_path.name}.json'
        if not ann_path.exists():
            continue
        ann  = json.loads(ann_path.read_text())
        tag  = ann['tags'][0]['name'] if ann['tags'] else 'no_tag'
        lbl  = label_map.get(tag)
        if lbl is not None:
            samples.append((img_path, lbl))
    return samples


# ── Rahman et al. — flat Tuberculosis / Normal folders ────────────────────────
def load_rahman(root):
    root, samples = Path(root), []
    for lbl, folder in [(1, 'Tuberculosis'), (0, 'Normal')]:
        d = root / folder
        if not d.exists():
            print(f'  Rahman: {d} not found — skipping')
            continue
        for ext in ('*.png', '*.jpg', '*.jpeg'):
            for p in sorted(d.glob(ext)):
                samples.append((p, lbl))
    return samples


# ── Shenzhen / Montgomery — filename suffix encodes label ─────────────────────
def _img_dir(root: Path, size: int) -> Path:
    cached = root / f'images_{size}'
    return cached if cached.exists() else root / 'images' / 'images'

def load_shenzhen(root, size=512):
    return [(p, int(p.stem.rsplit('_', 1)[-1]))
            for p in sorted(_img_dir(Path(root), size).glob('*.png'))]

def load_montgomery(root, size=512):
    return [(p, int(p.stem.rsplit('_', 1)[-1]))
            for p in sorted(_img_dir(Path(root), size).glob('*.png'))]


# ── CheXpert — CSV-driven, No Finding frontal images only ────────────────────
def load_chexpert_no_finding(root, n_samples=10_000, seed=42):
    root = Path(root)
    hits = list(root.rglob('train.csv'))
    if not hits:
        raise FileNotFoundError(f'train.csv not found under {root}')
    df = pd.read_csv(hits[0])
    if 'Frontal/Lateral' in df.columns:
        df = df[df['Frontal/Lateral'] == 'Frontal']
    df = df[df['No Finding'] == 1.0]
    n  = min(n_samples, len(df))
    df = df.sample(n, random_state=seed)
    print(f'  CheXpert: {len(df):,} qualifying images — sampling {n:,}')
    samples = []
    for _, row in df.iterrows():
        rel  = Path(row['Path'])
        path = root / rel
        if not path.exists():
            path = root / Path(*rel.parts[1:])   # strip leading dir prefix
        if path.exists():
            samples.append((path, 0))
    print(f'  CheXpert: resolved {len(samples):,} / {n:,} paths')
    return samples


# ── Unified dataset ───────────────────────────────────────────────────────────
class TBCXRDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform
        self.labels    = [s[1] for s in samples]

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

    def class_counts(self):
        pos = sum(self.labels)
        return len(self.labels) - pos, pos

## 6. Model

In [ ]:
class SharedEncoder(nn.Module):
    def __init__(self, backbone, pretrained=True):
        super().__init__()
        self.net = timm.create_model(backbone, pretrained=pretrained, features_only=True)
        self._channels = self.net.feature_info.channels()

    @property
    def out_channels(self): return self._channels[-1]

    def forward(self, x): return self.net(x)   # list of 5 feature maps


class DiagnosticHead(nn.Module):
    def __init__(self, in_channels, dropout=0.3):
        super().__init__()
        self.pool    = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(in_channels, 1)

    def forward(self, features):
        x = self.pool(features[-1]).flatten(1)
        return self.fc(self.dropout(x))


class TBDiagnosticModel(nn.Module):
    def __init__(self, encoder, head):
        super().__init__()
        self.encoder, self.head = encoder, head

    def forward(self, x): return self.head(self.encoder(x))


def build_model(backbone, pretrained=True, dropout=0.3):
    enc = SharedEncoder(backbone, pretrained)
    return TBDiagnosticModel(enc, DiagnosticHead(enc.out_channels, dropout))

## 7. Loss, metrics, and trainer

In [ ]:
class LabelSmoothBCE(nn.Module):
    def __init__(self, pos_weight=None, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, logits, targets):
        targets = targets.float() * (1 - self.smoothing) + 0.5 * self.smoothing
        return self.bce(logits.squeeze(1), targets)


def compute_auc(labels, probs):
    return roc_auc_score(labels, probs) if len(set(labels)) > 1 else float('nan')

def _sens_spec_curve(labels, probs):
    out = []
    for thr in np.linspace(0, 1, 200):
        p = (probs >= thr).astype(int)
        tp = ((p==1)&(labels==1)).sum(); fn = ((p==0)&(labels==1)).sum()
        tn = ((p==0)&(labels==0)).sum(); fp = ((p==1)&(labels==0)).sum()
        out.append((thr,
                    tp/(tp+fn) if (tp+fn) else 0.,
                    tn/(tn+fp) if (tn+fp) else 0.))
    return out

def find_who_tpp(labels, probs):
    cands = [(t,se,sp) for t,se,sp in _sens_spec_curve(labels,probs) if se >= 0.90]
    if not cands: return None
    t,se,sp = max(cands, key=lambda x: x[2])
    return {'threshold':t,'sensitivity':se,'specificity':sp,
            'meets_minimum_tpp':sp>=0.70,'meets_optimal_tpp':sp>=0.80}

def evaluate(labels, probs):
    la, pr = np.array(labels), np.array(probs)
    return {'auc': compute_auc(la, pr), 'who_tpp': find_who_tpp(la, pr)}

def print_report(metrics):
    print(f"  AUC : {metrics['auc']:.4f}")
    tpp = metrics['who_tpp']
    if tpp:
        t,se,sp = tpp['threshold'], tpp['sensitivity'], tpp['specificity']
        m = '\u2713' if tpp['meets_minimum_tpp'] else '\u2717'
        o = '\u2713' if tpp['meets_optimal_tpp'] else '\u2717'
        print(f'  WHO TPP (sens\u226590%): thr={t:.3f} sens={se:.3f} '
              f'spec={sp:.3f}  min(\u226570%)={m}  opt(\u226580%)={o}')
    else:
        print('  WHO TPP: no threshold reaches sensitivity >= 90%')


class Trainer:
    def __init__(self, model, optimizer, criterion, scheduler, device, output_dir):
        self.model, self.optimizer = model, optimizer
        self.criterion, self.scheduler = criterion, scheduler
        self.device     = device
        self.output_dir = Path(output_dir)
        self.scaler     = torch.amp.GradScaler(device.type)
        self.output_dir.mkdir(parents=True, exist_ok=True)

    # ── epoch loop ────────────────────────────────────────────────────────────
    def _run_epoch(self, loader, train):
        self.model.train(train)
        total_loss, all_probs, all_labels = 0., [], []
        with torch.set_grad_enabled(train):
            for imgs, labels in tqdm(loader, desc='train' if train else 'val', leave=False):
                imgs, labels = imgs.to(self.device), labels.to(self.device)
                with torch.amp.autocast(self.device.type):
                    logits = self.model(imgs)
                    loss   = self.criterion(logits, labels)
                if train:
                    self.scaler.scale(loss).backward()
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                    self.optimizer.zero_grad()
                total_loss += loss.item() * len(labels)
                probs = torch.sigmoid(logits).squeeze(1).float().detach().cpu().numpy()
                all_probs.append(probs)
                all_labels.append(labels.cpu().numpy())
        n = len(loader.dataset)
        return total_loss / n, np.concatenate(all_probs), np.concatenate(all_labels)

    # ── fit — auto-resumes from latest_checkpoint.pt if present ──────────────
    def fit(self, train_loader, val_loader, epochs, patience=10):
        best_auc, no_improve, start_epoch = 0., 0, 1

        latest = self.output_dir / 'latest_checkpoint.pt'
        if latest.exists():
            ckpt = torch.load(latest, map_location=self.device, weights_only=False)
            self.model.load_state_dict(ckpt['model'])
            self.optimizer.load_state_dict(ckpt['optimizer'])
            self.scheduler.load_state_dict(ckpt['scheduler'])
            start_epoch = ckpt['epoch'] + 1
            best_auc    = ckpt['best_auc']
            no_improve  = ckpt['no_improve']
            print(f'Resumed from epoch {ckpt["epoch"]}  '
                  f'best_auc={best_auc:.4f}  no_improve={no_improve}')
        else:
            print('Starting fresh training run')

        for epoch in range(start_epoch, epochs + 1):
            tr_loss, _, _              = self._run_epoch(train_loader, train=True)
            val_loss, val_probs, val_y = self._run_epoch(val_loader,   train=False)
            self.scheduler.step()

            metrics = evaluate(val_y, val_probs)
            print(f'\nEpoch {epoch:3d}  train={tr_loss:.4f}  val={val_loss:.4f}')
            print_report(metrics)

            if metrics['auc'] > best_auc:
                best_auc, no_improve = metrics['auc'], 0
                self._save_best(epoch, metrics)
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f'\nEarly stop at epoch {epoch} '
                          f'(no AUC gain for {patience} epochs).')
                    break

            # Save state every epoch so training can resume after interruption
            self._save_latest(epoch, metrics, best_auc, no_improve)

        print(f'\nBest val AUC: {best_auc:.4f}')

    # ── checkpoint helpers ────────────────────────────────────────────────────
    def _save_best(self, epoch, metrics):
        torch.save({'epoch': epoch, 'model': self.model.state_dict(),
                    'optimizer': self.optimizer.state_dict(),
                    'metrics': metrics},
                   self.output_dir / 'best_model.pt')
        print(f"  -> best checkpoint saved (AUC={metrics['auc']:.4f})")

    def _save_latest(self, epoch, metrics, best_auc, no_improve):
        torch.save({'epoch': epoch, 'model': self.model.state_dict(),
                    'optimizer': self.optimizer.state_dict(),
                    'scheduler': self.scheduler.state_dict(),
                    'metrics': metrics,
                    'best_auc': best_auc, 'no_improve': no_improve},
                   self.output_dir / 'latest_checkpoint.pt')

    def load_best(self):
        ckpt = torch.load(self.output_dir / 'best_model.pt',
                          map_location=self.device, weights_only=False)
        self.model.load_state_dict(ckpt['model'])
        e, a = ckpt['epoch'], ckpt['metrics']['auc']
        print(f'Loaded best model — epoch {e}  AUC={a:.4f}')

    def evaluate_loader(self, loader):
        _, probs, labels = self._run_epoch(loader, train=False)
        return probs, labels

## Stage 1 — Build and cache splits

Saved to `SPLITS_CACHE`. Re-run this cell only if you change dataset paths
or set `FORCE_REBUILD_SPLITS = True` in Config.

In [ ]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

def _build_splits():
    rng = np.random.default_rng(SEED)
    train_s, val_s, held_s = [], [], []

    def _split_85_15(samples, name):
        idx   = rng.permutation(len(samples))
        n_val = max(1, int(len(samples) * 0.15))
        val_s.extend(  [samples[i] for i in idx[:n_val]])
        train_s.extend([samples[i] for i in idx[n_val:]])
        pos = sum(1 for s in samples if s[1]==1)
        print(f'{name:12s} {pos} pos / {len(samples)-pos} neg — 85/15 split')

    if TBX11K:
        tr = load_tbx11k(TBX11K, split='train')
        vl = load_tbx11k(TBX11K, split='val')
        train_s.extend(tr); val_s.extend(vl)
        print(f'TBX11K       train {sum(1 for s in tr if s[1]==1)} pos / '
              f'{sum(1 for s in tr if s[1]==0)} neg  '
              f'| val {sum(1 for s in vl if s[1]==1)} pos')

    if RAHMAN:
        _split_85_15(load_rahman(RAHMAN), 'Rahman')

    if SHENZHEN:
        s = load_shenzhen(SHENZHEN, size=IMAGE_SIZE)
        if HELD_OUT == 'shenzhen':
            held_s.extend(s); print(f'Shenzhen     -> held-out ({len(s)})')
        else:
            _split_85_15(s, 'Shenzhen')

    if MONTGOMERY:
        s = load_montgomery(MONTGOMERY, size=IMAGE_SIZE)
        if HELD_OUT == 'montgomery':
            held_s.extend(s); print(f'Montgomery   -> held-out ({len(s)})')
        else:
            _split_85_15(s, 'Montgomery')

    if CHEXPERT:
        cx = load_chexpert_no_finding(CHEXPERT, n_samples=CHEXPERT_N_SAMPLES, seed=SEED)
        train_s.extend(cx)
        print(f'CheXpert     0 pos / {len(cx)} neg (all negative, train only)')

    tr_pos = sum(1 for s in train_s if s[1]==1)
    vl_pos = sum(1 for s in val_s   if s[1]==1)
    print(f'\nTotals  train: {tr_pos} pos / {len(train_s)-tr_pos} neg = {len(train_s)}')
    print(f'        val:   {vl_pos} pos / {len(val_s)-vl_pos} neg = {len(val_s)}')
    print(f'        held-out: {len(held_s)}')
    return train_s, val_s, held_s

cache_path = Path(SPLITS_CACHE)
loaded_from_cache = False

if not FORCE_REBUILD_SPLITS and cache_path.exists():
    print(f'Loading cached splits from {cache_path}')
    try:
        with open(cache_path, 'rb') as f:
            train_s, val_s, held_s = pickle.load(f)
        loaded_from_cache = True
        tr_pos = sum(1 for s in train_s if s[1]==1)
        vl_pos = sum(1 for s in val_s   if s[1]==1)
        print(f'train: {tr_pos} pos / {len(train_s)-tr_pos} neg = {len(train_s)}')
        print(f'val:   {vl_pos} pos / {len(val_s)-vl_pos} neg = {len(val_s)}')
        print(f'held-out: {len(held_s)}')
    except (EOFError, pickle.UnpicklingError) as e:
        print(f'Cache file corrupted ({e}) — deleting and rebuilding.')
        cache_path.unlink()

if not loaded_from_cache:
    print('Building splits from scratch...')
    train_s, val_s, held_s = _build_splits()
    # Atomic write: temp file + rename so a partial write never leaves a broken cache
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = cache_path.with_suffix('.tmp')
    with open(tmp, 'wb') as f:
        pickle.dump((train_s, val_s, held_s), f)
    tmp.rename(cache_path)
    print(f'Splits cached -> {cache_path}')

## Stage 2 — Model + optimiser setup

Re-run this cell (without Stage 1) to reset the model after a completed training run.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

def make_loader(samples, transform, weighted=False):
    ds = TBCXRDataset(samples, transform)
    sampler = None
    if weighted:
        lbls    = np.array(ds.labels)
        counts  = np.bincount(lbls)
        weights = (1.0 / counts)[lbls]
        sampler = WeightedRandomSampler(weights, len(weights))
    return DataLoader(ds, batch_size=BATCH_SIZE, sampler=sampler,
                      shuffle=(sampler is None), num_workers=NUM_WORKERS,
                      pin_memory=True), ds

train_loader, train_ds = make_loader(train_s, get_train_transforms(IMAGE_SIZE), weighted=True)
val_loader,   _        = make_loader(val_s,   get_val_transforms(IMAGE_SIZE))

model     = build_model(BACKBONE, pretrained=True, dropout=0.3).to(device)
criterion = LabelSmoothBCE(
    pos_weight=torch.tensor([POS_WEIGHT_SCALE]).to(device),
    smoothing=LABEL_SMOOTHING,
)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

neg, pos  = train_ds.class_counts()
print(f'Train: {pos} pos / {neg} neg  '
      f'(ratio {neg/pos:.1f}:1 — balanced in each batch by sampler)')
print(f'pos_weight={POS_WEIGHT_SCALE}  '
      f'(1.0 = symmetric loss, raise if sensitivity stays low)')

## Stage 3 — Train

**Auto-resume**: if `latest_checkpoint.pt` already exists in `OUTPUT_DIR`
(from a previous run that was interrupted), training picks up from the last
completed epoch automatically — just re-run this cell.

In [ ]:
trainer = Trainer(model, optimizer, criterion, scheduler, device, OUTPUT_DIR)
trainer.fit(train_loader, val_loader, epochs=EPOCHS, patience=PATIENCE)

## Stage 4 — Evaluate on validation set

Loads `best_model.pt` and runs a full evaluation pass.
Can be re-run independently of Stage 3.

In [ ]:
trainer.load_best()
val_probs, val_labels = trainer.evaluate_loader(val_loader)
print('\n── Validation set ──')
print_report(evaluate(val_labels, val_probs))

## Stage 5 — Held-out cross-source evaluation

Only runs if `HELD_OUT` is set to a source name in Config.

In [ ]:
if held_s:
    held_loader, _ = make_loader(held_s, get_val_transforms(IMAGE_SIZE))
    held_probs, held_labels = trainer.evaluate_loader(held_loader)
    print(f'\n── Held-out: {HELD_OUT} ({len(held_s)} images) ──')
    print_report(evaluate(held_labels, held_probs))
else:
    print('No held-out source — set HELD_OUT in Config to enable this.')